<a href="https://colab.research.google.com/github/Stephanie-0403/SGPA/blob/main/Trabajo%20excel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Herramienta de Procesamiento de Excel (Versión Simplificada)

Esta herramienta te permite subir un archivo de Excel, procesarlo automáticamente para corregir los formatos de datos y descargar la versión limpia. Sigue estos dos sencillos pasos:

---

### **Paso 1: Sube tu archivo de Excel**

Ejecuta la siguiente celda para subir tu archivo. Te aparecerá un botón que dice 'Elegir archivos'.

In [ ]:
from google.colab import files
import pandas as pd
import io

# *** NO MODIFICAR ESTA CELDA ***

# Sube todos los archivos de Excel
print("Haz clic en 'Elegir archivos' y selecciona TODOS tus archivos de Excel a combinar.")
uploaded = files.upload()

# Obtiene los nombres de todos los archivos subidos
uploaded_file_names = list(uploaded.keys())

print(f"\nArchivos subidos: {', '.join(uploaded_file_names)}. Ahora ve al Paso 2 para procesarlos.")

Haz clic en 'Elegir archivos' y selecciona tu archivo de Excel.


---

### **Paso 2: Procesar y Descargar el Archivo Limpio**

Ahora ejecuta la siguiente celda. El programa leerá tu archivo, aplicará las correcciones de formato y automáticamente te ofrecerá descargar el archivo procesado.



In [ ]:
import functools

# *** NO MODIFICAR ESTA CELDA ***

print("Iniciando procesamiento y consolidación de datos...")

all_dfs = []
# Carga cada archivo Excel subido en un DataFrame de pandas
for file_name in uploaded_file_names:
    try:
        df_temp = pd.read_excel(io.BytesIO(uploaded[file_name]))

        # *** NUEVA LÓGICA PARA MANEJAR 'id' en lugar de 'Folio' ***
        if 'id' in df_temp.columns and 'Folio' not in df_temp.columns:
            df_temp = df_temp.rename(columns={'id': 'Folio'})
            print(f"  - Archivo '{file_name}': Columna 'id' renombrada a 'Folio'.")
        # *** FIN DE LA NUEVA LÓGICA ***

        all_dfs.append(df_temp)
        print(f"  - Archivo '{file_name}' cargado exitosamente. Filas: {len(df_temp)}")
    except Exception as e:
        print(f"  - ERROR al cargar '{file_name}': {e}. Este archivo será omitido.")

if not all_dfs:
    print("No se pudieron cargar archivos o no se subieron. Por favor, revisa el Paso 1.")
elif len(all_dfs) == 1: # Si solo se subió un archivo, no es necesario fusionar
    df = all_dfs[0]
    print("\n--- Solo se subió un archivo; no se realizó ninguna fusión. ---")
else:
    print("\n--- Consolidando DataFrames ---")
    # Usa el primer DataFrame como base y fusiona los demás
    # Asume 'Folio' como la columna clave para la fusión.
    # Usa 'outer' merge para mantener todos los folios de todos los archivos.
    # Asegurarse de que 'Folio' exista en el DataFrame antes de intentar fusionar
    # Filtra DataFrames que no tienen la columna 'Folio' para evitar errores en merge
    dfs_with_folio = [d for d in all_dfs if 'Folio' in d.columns]

    if not dfs_with_folio:
        print("Error: Ninguno de los archivos subidos contiene la columna 'Folio' o 'id' para la fusión. No se puede consolidar.")
        # Opcional: Salir o manejar de otra forma
        df = pd.DataFrame() # Crear un DataFrame vacío para evitar errores posteriores
    else:
        df_merged = functools.reduce(lambda left, right: pd.merge(left, right, on='Folio', how='outer', suffixes=('_left', '_right')),
                                     dfs_with_folio)
        df = df_merged # Renombra a df para usar el código de limpieza existente
        print(f"DataFrames consolidados. Filas totales después de la fusión: {len(df)}")

if not df.empty:
    print("\n--- Vista Previa del Archivo Consolidado Original ---")
    display(df.head())

    print("\n--- Información de Tipos de Datos del Consolidado Original ---")
    display(df.info())

    # --- Corrección de tipos de datos (aplicada al DataFrame fusionado) ---

    # 1. Convertir columnas de fecha
    fecha_cols = [
        'Desde', 'Hasta', 'Fecha solicitud', 'Fecha liberacion analista',
        'FECHA LIB COORD, GERENCIA Y DIC', 'Fecha para rev'
    ]
    for col in fecha_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # 2. Convertir columnas que deberían ser texto
    texto_cols = ['Estatus', 'Estado de folio', 'Concluido flujo', 'Concepto', 'Descripción', 'NC Aplicar']
    for col in texto_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)

    # 3. Formatear columnas numéricas
    numerico_cols = ['Presupuesto solicitado', 'Presupuesto ejecutado', 'Por ejecutar', 'Por ejecutar sin IVA']
    for col in numerico_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    print("\n--- Procesamiento de datos completado. ---")

    print("\n--- Vista Previa del Archivo Procesado ---")
    display(df.head())

    print("\n--- Información de Tipos de Datos Después del Procesamiento ---")
    display(df.info())

    # Define un nuevo nombre para el archivo procesado
    output_file_name_processed = f"consolidado_limpio.xlsx"

    # Guarda el DataFrame limpio en un nuevo archivo Excel
    df.to_excel(output_file_name_processed, index=False)

    # Descarga el archivo
    print(f"\n¡Descargando el archivo '{output_file_name_processed}' ahora mismo!")
    files.download(output_file_name_processed)

    print("\nProceso finalizado. El archivo ha sido descargado.")
else:
    print("No hay datos para procesar o consolidar. El archivo final no se generará.")

### 1. Sube tu archivo de Excel a Colab

Ejecuta la siguiente celda. Te pedirá que elijas un archivo desde tu computadora para subirlo a la sesión de Colab. Una vez subido, estará disponible para que Python lo lea.

In [ ]:
from google.colab import files
import pandas as pd
import io

# Sube el archivo de Excel
uploaded = files.upload()

# El nombre del archivo subido será la clave en el diccionario 'uploaded'
file_name = next(iter(uploaded))

print(f"Archivo '{file_name}' subido exitosamente.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Lee el archivo de Excel y corrige los tipos de datos

Una vez subido, puedes leerlo con Pandas. Aquí voy a asumir que tu hoja de datos principal está en la primera pestaña del Excel. Luego, aplicaremos las correcciones de formato que mencionaste.

In [ ]:
# Carga el archivo Excel en un DataFrame de pandas
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print("Primeras 5 filas del DataFrame original:")
display(df.head())

print("\nInformación de los tipos de datos originales:")
display(df.info())

# --- Corrección de tipos de datos ---

# 1. Convertir columnas de fecha ('Desde', 'Hasta', 'Fecha solicitud', 'Fecha liberacion analista', 'FECHA LIB COORD, GERENCIA Y DIC', 'Fecha para rev')
# Usamos 'coerce' para convertir valores que no son fechas en NaT (Not a Time), lo que ayuda a identificar errores.
for col in ['Desde', 'Hasta', 'Fecha solicitud', 'Fecha liberacion analista', 'FECHA LIB COORD, GERENCIA Y DIC', 'Fecha para rev']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"Columna '{col}' convertida a tipo fecha.")
    else:
        print(f"Advertencia: Columna '{col}' no encontrada para conversión de fecha.")

# 2. Convertir columnas que deberían ser texto ('Estatus', 'Estado de folio', 'Concluido flujo')
# Convertir a string para asegurar que no se interpreten como números si contienen texto mezclado o solo texto.
for col in ['Estatus', 'Estado de folio', 'Concluido flujo']:
    if col in df.columns:
        df[col] = df[col].astype(str)
        print(f"Columna '{col}' convertida a tipo texto.")
    else:
        print(f"Advertencia: Columna '{col}' no encontrada para conversión a texto.")

# 3. Formatear columnas numéricas (ej. 'Presupuesto solicitado', 'Presupuesto ejecutado', 'Por ejecutar', 'Por ejecutar sin IVA')
# Asegurar que sean numéricas y manejar posibles errores
for col in ['Presupuesto solicitado', 'Presupuesto ejecutado', 'Por ejecutar', 'Por ejecutar sin IVA']:
    if col in df.columns:
        # Convertir a numérico, convirtiendo errores a NaN (Not a Number)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"Columna '{col}' convertida a tipo numérico.")
    else:
        print(f"Advertencia: Columna '{col}' no encontrada para conversión numérica.")

print("\nPrimeras 5 filas del DataFrame después de la limpieza:")
display(df.head())

print("\nInformación de los tipos de datos después de la limpieza:")
display(df.info())

print("\nSe han aplicado las correcciones de formato a las columnas especificadas.")

### 3. Descarga el archivo procesado

Si deseas guardar el archivo con los cambios aplicados, puedes descargarlo de nuevo a tu computadora.

In [ ]:
# Define un nuevo nombre para el archivo procesado
output_file_name = f"procesado_{file_name}"

# Guarda el DataFrame limpio en un nuevo archivo Excel
df.to_excel(output_file_name, index=False)

# Descarga el archivo
files.download(output_file_name)

print(f"Archivo '{output_file_name}' descargado exitosamente.")